In [1]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, first install it by visiting [https://ollama.com/download](https://ollama.com/download) and selecting your operating system. Once installed, open a terminal and type `ollama run llama3`. This will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test the local Ollama server, run `curl http://localhost:11434` in your terminal. You should see a JSON response similar to `{"models": [...]}`.\n\nYou can also install the Python client with `pip install ollama` and use it in your Python scripts.'

In [5]:
assistant.rag("How do I run Olama locally?")

'This FAQ database does not contain information on how to run Olama locally.'

In [8]:
from google.genai import types
from google import genai

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"]
    }
}

tools = types.Tool(function_declarations=[search_tool])
config = types.GenerateContentConfig(tools=[tools])

messages = "I just discovered the course. Can I join it?"
answer = gemini_client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=messages,
    config=config
)

print(answer.text)

None
